In [1]:
from Functions_wrapped import *
import matplotlib.pyplot as plt
from scipy import optimize as opt
from scipy import stats as sts
from BG2_functions import *
import time


In [2]:
ls_ps=[0.2,0.1,0.05,0.02,0.01,0.005,0.002,0.001,0.0005,0.0002,0.0001]
ls_N=[]
ls_D=[]
rows=[]
for pi in ls_ps:
    bbg2= better_BG2(pi)
    EGS=bbg2[0]
    rows.append({'Prevalence': pi, 'EGS': EGS})
    new_N=EGS*10
    new_D=np.ceil(new_N*pi)
    new_n=np.round(new_D/pi)
    ls_N.append(int(new_n))
    ls_D.append(int(new_D))
bg2_df = pd.DataFrame(rows)

In [ ]:

M_id=12
# ── User-defined parameters ──────────────────────────────────────────────────
N_list    =ls_N[:M_id]# [500, 200, 500, 1000]   # sample sizes to sweep
diff_list =ls_D[:M_id]# [25,   2,  1,   1]   # differentiation level for each N (same length as N_list)

checks_hierarchical = 500        # MC checks for hierarchical simulation
metrics_checks      = 1e0        # max_checks passed to mean_metrics_fast (internal scaler=1e3)

# Random design parameters (from rand_WA_wrapped.py defaults)
rand_guesses          = 2    # number of random WA candidates to evaluate
rand_max_redundancy   = 2.0   # upper bound on well-redundancy relative to N*log2(N)
rand_min_redundancy   = 0.5   # lower bound on well-redundancy
rand_n_compounds_per_well = 0 # 0 = auto-select
rand_n_wells          = 0     # 0 = auto-select
rand_max_compounds    = 0     # 0 = auto-select (uses get_max_C default)
# ────────────────────────────────────────────────────────────────────────────

rows = []

for N, diff in zip(N_list, diff_list):
    # Maximum number of multidim dimensions that make sense for this N
    # (require L1 = ceil(N^(1/k)) >= 2  =>  k <= log2(N))
    max_dims = max(3, int(np.floor(np.log2(max(N, 2)))))

    # ── Build list of (method_name, assign_fn, kwargs) for deterministic methods
    methods_spec = []

    # Matrix (multidim-2 alias)
    methods_spec.append(('Matrix', assign_wells_mat, {'n_compounds': N}))

    # Multidim 3 … max_dims
    nd=3
    methods_spec.append((f'multidim-{nd}', assign_wells_multidim,
                            {'n_compounds': N, 'n_dims': nd}))

    # Binary
    methods_spec.append(('Binary', assign_wells_bin,
                         {'n_compounds': N, 'differentiate': diff}))


    # ── Deterministic methods: time WA construction + metrics calculation ─────
    for method_name, wa_fn, wa_kwargs in methods_spec:
        t0 = time.perf_counter()
        WA = wa_fn(**wa_kwargs)
        result = mean_metrics_fast(well_assigner=WA, differentiate=diff,
                                   max_checks=metrics_checks)
        elapsed = time.perf_counter() - t0
        rows.append({'N': N, 'diff': diff, 'Method': method_name,'Prevalence': diff/N,
                     'time': elapsed, 'mean_tests': result[0]})

    # ── Random: assign_wells_random_precomp already returns mean_tests ────────
    if False:
        t0 = time.perf_counter()
        _, rand_mean_tests, _ = assign_wells_random_precomp(
            n_compounds=N,
            differentiate=diff,
            guesses=rand_guesses,
            max_redundancy=rand_max_redundancy,
            min_redundancy=rand_min_redundancy,
            n_compounds_per_well=rand_n_compounds_per_well,
            n_wells=rand_n_wells,
            max_compounds=rand_max_compounds,
            return_me=True,
        )
        elapsed = time.perf_counter() - t0
        rows.append({'N': N, 'diff': diff, 'Method': 'Random',
                    'time': elapsed, 'mean_tests': rand_mean_tests})

    # ── Hierarchical: calculate_full_metrics_hierarchical_fast ────────────────
    t0 = time.perf_counter()
    hier = calculate_metrics_hierarchical_fast(N, diff, checks=checks_hierarchical, 
                                               keep_ratios_constant=True, Faster=True)
    elapsed = time.perf_counter() - t0
    # hier[0] = mean total experiments
    rows.append({'N': N, 'diff': diff, 'Method': 'Hierarchical','Prevalence': diff/N, 
                 'time': elapsed, 'mean_tests': hier[0]})

timing_df = pd.DataFrame(rows, columns=['N', 'diff', 'Prevalence', 'Method', 'time', 'mean_tests'])
timing_df['ET'] = timing_df['mean_tests'] / timing_df['N']
timing_df[['N', 'diff','Method', 'Prevalence',  'ET', 'time']]



In [ ]:
timing_df.to_csv("timing_results_PoolPy.csv", index=False)


In [3]:
rows = []

for p in ls_ps:
    t0 = time.perf_counter()
    #DM=int(np.max([np.log2(1/p),9])+1)
    DM=50
    result=brute_better_BG2(p,DM)
    elapsed = time.perf_counter() - t0
    rows.append({'Method': 'BG_2','Prevalence': p,
                    'time': elapsed, 'mean_tests': result[-1], 'size': len(result[0])})
    t0 = time.perf_counter()
    result=brute_BG2_Gen_MD(p,DM)
    elapsed = time.perf_counter() - t0
    rows.append({'Method': 'BG_2_MD','Prevalence': p,
                    'time': elapsed, 'mean_tests': result[-1], 'size': result[0]})

timing_df = pd.DataFrame(rows, columns=['Prevalence', 'Method', 'time', 'mean_tests', 'size'])
timing_df['ET'] = timing_df['mean_tests']
timing_df[['Method', 'Prevalence', 'size',  'ET', 'time']]



/Users/ltalamanca/My Drive/Git/PoolPy/BG2_functions.py:135: RuntimeWarning: divide by zero encountered in scalar divide
  return((D*N+np.sum(ev))/(N**D))


,Method,Prevalence,size,ET,time
0,BG_2,0.2000,1,0.821333,0.012254
1,BG_2_MD,0.2000,2,0.889244,0.038125
2,BG_2,0.1000,2,0.586304,0.040434
3,BG_2_MD,0.1000,3,0.670098,0.032241
4,BG_2,0.0500,2,0.376986,0.004034
5,BG_2_MD,0.0500,2,0.426823,0.027027
6,BG_2,0.0200,3,0.197977,0.005674
7,BG_2_MD,0.0200,2,0.249976,0.027920
8,BG_2,0.0100,4,0.117908,0.007172
9,BG_2_MD,0.0100,3,0.167767,0.029351


In [4]:
timing_df.to_csv("timing_results_our_BG2.csv", index=False)

In [5]:
calculate_full_metrics_hierarchical_fast(100,10,checks=1000)

[59.507, 10, [11, 3], 100, 48.507, 3, 107.151, -47.644]

In [6]:

from Functions_wrapped import *

In [ ]:
def full_iterative_uneven_splitter(id_samps, id_positives, ratios):
    if len(ratios)==1:
        ratio=ratios[0]
        ratios=[np.inf]
    else:
        ratio=ratios[0]
        ratios=ratios[1:]

    if len(id_samps)<=ratio:
        return len(id_samps),len(id_positives), len(id_samps)-len(id_positives)

    pools=list(split(id_samps, ratio))
    partials=0
    n_pos=0
    n_neg=0
    for pool in pools:
        if len(set(pool).intersection(id_positives))>0:
            new_partials, new_n_pos, new_n_neg=full_iterative_uneven_splitter(pool,set(pool).intersection(id_positives),ratios)
            partials+=new_partials
            n_pos+=new_n_pos+1
            n_neg+=new_n_neg
        else:
            n_neg+=1
    if n_pos<0 or n_neg<0:
        print(n_pos, n_neg)
    return ratio+partials, n_pos, n_neg

In [9]:
def calculate_full_metrics_hierarchical_fast(n_compounds,  differentiate:int, checks=1e4, keep_ratios_constant=False,  **kwargs):
    id_samps=np.arange(n_compounds)
    details={}
    posiz=pick_rand_pos(n_compounds, differentiate, checks)
    BM=[[0],np.inf]
    FNN=0
    FNP=0
    
    if 'ls_splits' in kwargs.keys():
        list_splits=[kwargs['ls_splits']]
    else:
        list_splits=uneven_wrapper(n_compounds, differentiate)
    ls_id=0
    for splito in list_splits:
        NP=0
        FM=0
        NPos=0
        NNeg=0
        for id_pos in posiz:
            posx=np.array(id_pos)
            measures=full_iterative_uneven_splitter(id_samps,posx,splito)
            FM+=measures[0]
            NPos+=measures[1]
            NNeg+=measures[2]
            NP+=1
                
        layers=len(splito)+1
        MC=int(np.ceil(n_compounds/splito[0]))
        #details.update({ls_id:[FM/NP, MC, splito, int(np.round((NP-1)/(NP),2)*100), FM/NP-splito[0],layers]})
        ls_id+=1
        if FM/NP<BM[1]:
            BM=[splito,FM/NP]
            FNP=NPos/NP
            FNN=NNeg/NP

    layers=len(BM[0])+1
    MC=int(np.ceil(n_compounds/BM[0][0]))
    return([BM[1], MC, BM[0], int(np.round((NP-1)/(NP),2)*100), BM[1]-BM[0][0],layers,FNP, FNN])

In [10]:
calculate_full_metrics_hierarchical_fast(100,10,checks=100)

11 -1
22 -3
35 -4
11 -1
22 -4
35 -5
22 -3
11 -1
35 -4
108 -13
22 -3
22 -3
46 -6
22 -3
23 -2
11 -1
22 -4
35 -5
107 -13
11 -1
22 -3
35 -4
11 -1
24 -1
11 -1
74 -5
11 -1
22 -3
35 -4
22 -3
11 -1
35 -4
11 -1
85 -8
22 -3
23 -2
22 -4
23 -3
22 -3
22 -4
46 -7
95 -12
11 -1
22 -3
35 -4
22 -3
23 -2
11 -1
11 -1
24 -2
85 -8
11 -1
24 -1
22 -3
11 -1
35 -4
11 -1
22 -4
35 -5
97 -10
11 -1
22 -3
35 -4
11 -1
24 -1
11 -1
22 -4
35 -5
97 -10
22 -3
22 -4
46 -7
73 -5
22 -3
23 -2
22 -3
11 -1
35 -4
11 -1
11 -1
24 -2
85 -8
22 -3
11 -1
35 -4
11 -1
24 -1
22 -3
11 -1
35 -4
97 -9
22 -3
35 -3
11 -1
24 -1
11 -1
22 -4
35 -5
97 -9
22 -3
23 -2
11 -1
22 -4
35 -5
11 -1
24 -1
85 -8
22 -3
11 -1
35 -4
22 -3
23 -2
11 -1
22 -4
35 -5
96 -11
11 -1
22 -3
11 -1
35 -4
11 -1
11 -1
24 -2
74 -6
22 -3
22 -3
46 -6
22 -4
23 -3
22 -3
22 -4
46 -7
118 -16
11 -1
24 -1
22 -3
22 -4
46 -7
85 -7
22 -3
35 -3
22 -4
35 -4
11 -1
11 -1
24 -2
97 -9
22 -3
35 -3
11 -1
24 -1
22 -4
23 -3
85 -7
22 -3
23 -2
11 -1
22 -4
35 -5
73 -6
11 -1
22 -3
35 -4
22 -3
22 -4


KeyboardInterrupt: 

In [ ]:
better_BG2_Gen(0.0001,5)

In [ ]:
better_BG2_Gen(0.0001,30)

In [ ]:
brute_BG2_Gen_MD(0.2,3)

In [ ]:
brute_BG2_Gen_MD(1e-4, 30)

In [ ]:
brute_BG2_Gen_MD(0.0001, 50)


In [ ]:
11**4

In [ ]:
(1-1e-4)**(11**4)

In [ ]:
sts.binom.pmf(2,11**4, 1e-4)

In [ ]:
brute_BG2_Gen_MD(0.0001, 9)

In [ ]:
better_BG2_Gen(0.02,3)

In [ ]:
a=[1,2]
b=[4,0]

In [ ]:
np.max([a,b], 0)

In [ ]:

def ET_BG2_Gen_opt(N, p):
    Mm=np.cumprod(N)
    M=np.flip(np.cumprod(np.flip(N)))
    Mm=np.insert(Mm, 0, 1, axis=0)
    M=np.insert(M, len(M), 1, axis=0)
    N=np.insert(N, 0, 1, axis=0)
    print(Mm)
    psi=1-(1-p)**M
    print(M)
    print(psi)
    print((1-(1-psi[:-1])**Mm[:-1]))
    print((1-psi[:-1])**Mm[:-1])
    print(Mm[1:]*psi[:-1]/(1-(1-psi[:-1])**Mm[:-1]))
    return((1+psi[0]*(np.sum(Mm[1:]*psi[:-1]/(1-(1-psi[:-1])**Mm[:-1]))) )/M[0])
